In [ ]:
!pip install datasets transformers bitsandbytes peft accelerate trl flash-attn --no-build-isolation

In [ ]:
import logging
from typing import Union, Optional
from dataclasses import dataclass
from collections.abc import Mapping

import numpy as np
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader

from datasets import load_dataset, Dataset
from accelerate import Accelerator

from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    PreTrainedTokenizerBase,
)
from transformers.data.data_collator import DataCollatorMixin

accelerator = Accelerator()
logger = logging.getLogger()

# 1) Supervised Training

![](https://substackcdn.com/image/fetch/$s_!HgRZ!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Ffb9d0144-3952-42db-8382-8e2eb37d917e_1670x640.png)

# 🎓 Supervised Fine-Tuning (SFT) for Large Language Models

---

Supervised Fine-Tuning (SFT) is the process of training a pretrained language model on **(instruction, response)** pairs so it learns to follow instructions and behave as a helpful assistant. This is typically the **first step after pretraining** in the LLM training pipeline:

> **Pretraining** → **SFT** → RLHF / DPO (alignment)

## 1.1 Tokenization

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chat templates convert a list of role/content dicts into the token format
# the model expects (special tokens like <|im_start|>, role tags, etc.)
messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]

# Input: list[dict] -> Output: (1, seq_len) token IDs
tokenized_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)
print(tokenizer.decode(tokenized_chat[0]))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<|im_start|>system
You are a friendly chatbot who always responds in the style of a pirate<|im_end|>
<|im_start|>user
How many helicopters can a human eat in one sitting?<|im_end|>
<|im_start|>assistant



In [ ]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k")
dataset = dataset.remove_columns("prompt")
sample_size = int(len(dataset["train_sft"]) * 0.1)
dataset["train_sft"] = dataset["train_sft"].select(range(sample_size))
print(dataset["test_sft"]["messages"][100])

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

[{'content': "Can you explain how Henri Matisse's experience with illness influenced his later works?", 'role': 'user'}, {'content': "Henri Matisse's experience with illness significantly influenced his later works, particularly in the way that he approached art. In 1941, Matisse was diagnosed with cancer and was confined to a wheelchair as a result. This experience provoked him to explore new creative avenues with his art, both in style and the tools he used to create.\n\nMatisse began to experiment with a new technique known as paper cut-outs. He used scissors to cut shapes and colors from colored paper and then arranged them in a composition. The bright and vibrant colors that Matisse used in these cut-outs were a direct contrast to the muted palette he had been employing before his illness. The vibrant hues represented a sense of vitality that Matisse was seeking, which he believed could cure the sadness and despair that he felt during his early stages. His cut-outs became some of 

## 1.2 Quantization

# 🧊 Understanding `BitsAndBytesConfig` for Quantization

---

### Why quantize?

A 7B parameter model in fp16 needs **~14GB** of GPU memory just for weights. Quantization reduces this by storing weights in lower precision.

| Precision | Bits per param | Memory for 7B model |
|-----------|---------------|---------------------|
| fp32 | 32 bits | ~28GB |
| fp16 / bf16 | 16 bits | ~14GB |
| 8-bit | 8 bits | ~7GB |
| 4-bit | 4 bits | ~3.5GB |

---

### `load_in_4bit=True` — The main switch

This tells BitsAndBytes to **load the model weights in 4-bit precision**. But *how* those 4 bits are used matters a lot — that's what the other configs control.

> Think of it like saying "I want a compressed image." OK, but **what compression algorithm?** JPEG? PNG? WebP? That's what the sub-configs decide.

---

### Breaking down each config:
```python
BitsAndBytesConfig(
    load_in_4bit=True,               # ① WHAT: load weights in 4-bit
    bnb_4bit_quant_type="nf4",       # ② HOW: which 4-bit format to use
    bnb_4bit_compute_dtype=torch.float16,  # ③ COMPUTE: what precision for math
    bnb_4bit_use_double_quant=False,       # ④ EXTRA: compress even further?
)
```

---

### ① `load_in_4bit=True` — Load weights in 4-bit

The on/off switch. Weights are quantized from fp16/fp32 → 4-bit when loading.
```
Without:  each weight = 16 bits  →  7B × 2 bytes = ~14GB
With:     each weight = 4 bits   →  7B × 0.5 bytes = ~3.5GB
```

---

### ② `bnb_4bit_quant_type="nf4"` — Which 4-bit format?

4 bits = 16 possible values. But **how you distribute** those 16 values matters:
```
"fp4" (float4):
  16 values evenly spaced across the range
  ├──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┼──┤
  Evenly spaced — but most weights cluster near zero!

"nf4" (NormalFloat4):
  16 values spaced according to a NORMAL DISTRIBUTION
  ├┼┼┼──┼────┼────────┼────────┼────┼──┼┼┼┤
  More values near zero where most weights actually are!
```

> 🔑 Pretrained weights follow a roughly normal distribution (most near zero, few large values). **NF4 matches this distribution**, so it preserves more information with the same 4 bits. Always use `"nf4"` — it's strictly better than `"fp4"`.

---

### ③ `bnb_4bit_compute_dtype=torch.float16` — Precision during computation

Weights are **stored** in 4-bit, but you can't do matrix multiplication in 4-bit. During the forward pass, weights are **dequantized on-the-fly** to a higher precision for actual computation:
```
Storage:     4-bit  (saves memory)
                ↓
Dequantize:  → fp16 or bf16  (for actual matmul)
                ↓
Compute:     x @ W  in fp16/bf16  (normal math)
```

| Option | Speed | Quality | When to use |
|--------|-------|---------|-------------|
| `torch.float16` | Fast | Good | Default, most GPUs |
| `torch.bfloat16` | Fast | Better for large values | Ampere+ GPUs (A100, RTX 3090+) |
| `torch.float32` | Slow | Best | Debugging only |

---

### ④ `bnb_4bit_use_double_quant=False` — Quantize the quantization constants?

When you quantize weights, you store **scaling constants** (one per block of ~64 weights). These constants themselves take memory.
```
Double quant OFF:
  Weights: 4-bit  |  Scaling constants: fp32 (32 bits each)
  Total: ~4.5 bits per param

Double quant ON:
  Weights: 4-bit  |  Scaling constants: ALSO quantized to 8-bit
  Total: ~4.1 bits per param  (saves ~0.4 bits/param)
```

> Use `True` when memory is very tight. The quality loss is negligible.

---

### 8-bit is simpler — no sub-configs needed
```python
BitsAndBytesConfig(load_in_8bit=True)
# That's it! 8-bit quantization has fewer knobs to tune.
# Less aggressive compression, but also less quality loss.
```

---

### Other model loading configs:
```python
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",              # automatically place layers across available GPUs
    torch_dtype=torch_dtype,        # default dtype for NON-quantized params (layernorm, etc.)
    use_cache=False,                # disable KV cache — incompatible with gradient checkpointing
    quantization_config=bnb_config, # apply the quantization config above
)
```

| Config | What it does | When to change |
|--------|-------------|----------------|
| `device_map="auto"` | Auto-distributes model layers across GPUs/CPU | Use `{"": 0}` for single GPU |
| `torch_dtype` | Precision for non-quantized params (layernorm, biases) | `bf16` for Ampere+, `fp16` otherwise |
| `use_cache=False` | Disables KV cache (stores past key/values for faster generation) | Must be `False` during training with gradient checkpointing |
| `quantization_config` | The `BitsAndBytesConfig` from above | Only needed for quantized training |

---

> 💡 **Summary:** `load_in_4bit` says **"compress to 4 bits"**, but the sub-configs control **how** to compress (`nf4`), **what precision to compute in** (`fp16`), and **whether to compress further** (`double_quant`). It's like choosing between JPEG quality levels — same file format, different quality/size tradeoffs.

In [ ]:
def get_quantization_config(load_in_4bit=False, load_in_8bit=False):
    """Return a BitsAndBytesConfig for 4-bit or 8-bit quantization, or None."""
    if load_in_4bit:
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",       # normalized float 4-bit
            bnb_4bit_use_double_quant=False,
        )
    elif load_in_8bit:
        return BitsAndBytesConfig(load_in_8bit=True)
    return None

torch_dtype = torch.bfloat16
quantization_config = get_quantization_config(load_in_4bit=True)

## 1.4 Model Loading

In [ ]:
def get_kbit_device_map():
    """Map quantized model to the correct device (GPU index or CPU)."""
    if torch.cuda.is_available():
        return {"": Accelerator().local_process_index}
    return None

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch_dtype,
    use_cache=False, # disable KV cache for training
    quantization_config=quantization_config,
)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

## 1.5 Model Training

In [ ]:
# LoRA injects small trainable rank-decomposition matrices into attention layers,
# drastically reducing the number of trainable parameters.

peft_config = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [ ]:
training_args = SFTConfig(
    bf16=False,
    gradient_accumulation_steps=128,
    learning_rate=2.0e-05,
    log_level="info",
    logging_steps=5,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    max_steps=-1,
    num_train_epochs=1,
    output_dir="data/zephyr-7b-sft-lora",
    overwrite_output_dir=True,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=2,
    seed=42,
    packing=True,
)

/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:453: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:495: UserWarning: You are using packing, but the attention implementation is not set to 'flash_attention_2' or 'kernels-community/vllm-flash-attn3'. Packing flattens batches into a single sequence, and Flash Attention is the only known attention mechanisms that reliably support this. Using other implementations may lead to cross-contaminatio

Tokenizing train dataset:   0%|          | 0/20786 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/20786 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/23110 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/23110 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train_sft"],
    eval_dataset=dataset["test_sft"],
    processing_class=tokenizer,
)

train_result = trainer.train()
print(train_result.metrics)

## 1.6 Evaluation and Model Saving

In [ ]:
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)
logger.info(f"Model saved to {training_args.output_dir}")

Saving model checkpoint to data/zephyr-7b-sft-lora
Configuration saved in data/zephyr-7b-sft-lora/config.json
Configuration saved in data/zephyr-7b-sft-lora/generation_config.json
Model weights saved in data/zephyr-7b-sft-lora/model.safetensors
chat template saved in data/zephyr-7b-sft-lora/chat_template.jinja
tokenizer config file saved in data/zephyr-7b-sft-lora/tokenizer_config.json
Special tokens file saved in data/zephyr-7b-sft-lora/special_tokens_map.json
chat template saved in data/zephyr-7b-sft-lora/chat_template.jinja
tokenizer config file saved in data/zephyr-7b-sft-lora/tokenizer_config.json
Special tokens file saved in data/zephyr-7b-sft-lora/special_tokens_map.json


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, training_args.output_dir)
# produces a standard model with LoRA baked in
model = model.merge_and_unload()

# 2 Two Dataset Formats for SFT

SFT datasets come in two formats. The preprocessing and collation differ for each:

| Format | Structure | Use case |
|--------|-----------|----------|
| **Conversational** | `{"messages": [{"role": "user", "content": ...}, {"role": "assistant", "content": ...}]}` | Multi-turn chat data |
| **Completion** | `{"prompt": "...", "completion": "..."}` | Single-turn instruction/response pairs |
```python
# Conversational format
{"messages": [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
]}

# Completion format
{"prompt": "What is 2 + 2?", "completion": "2 + 2 equals 4."}

# Instruction Example
{
    "instruction": "Explain photosynthesis",
    "output": "Photosynthesis is the process by which plants convert sunlight into energy."
}
```

---

## 💬 Chat Templates: Why They Matter

Different models expect different special token formats. Chat templates handle this automatically:
```python
# Input: structured messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]

# Output (Qwen format): formatted string with special tokens
# <|im_start|>system
# You are a helpful assistant.<|im_end|>
# <|im_start|>user
# What is 2+2?<|im_end|>
# <|im_start|>assistant
```

---

## 🎯 Completion-Only Loss Masking

The key idea in SFT: **don't penalize the model for the prompt — only train it on the response.**
```python
text = "### Question: What is 2+2? ### Answer: 2+2 equals 4."

# DataCollatorForLanguageModeling (standard):
labels = [Question, What, is, 2+2?, Answer:, 2+2, equals, 4.]
#         ^^^^^^^^^^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^^^^^^^^^
#         loss computed on ALL tokens (including the question)

# DataCollatorForCompletionOnlyLM (completion-only):
labels = [-100, -100, -100, -100, -100, 2+2, equals, 4.]
#         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^
#         prompt MASKED (ignored)       loss ONLY on response
```

> This ensures the model learns **what to answer**, not **how to repeat the question**.

## 2.1 Tokenization

Example of Chat Template in Model Repo: [openai/gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b/blob/main/chat_template.jinja)

```python
# Example chat template for conversational datasets
CHAT_TEMPLATE = """{%- if tools %}
{{- '<|im_start|>system\n' }}
{%- if messages[0]['role'] == 'system' %}
{{- messages[0]['content'] }}
{%- else %}
{{- 'You are a helpful assistant.' }}
{%- endif %}
{{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
{%- for tool in tools %}
{{- "\n" }}
{{- tool | tojson }}
{%- endfor %}
{{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
{%- if messages[0]['role'] == 'system' %}
{{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
{%- else %}
{{- '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n' }}
{%- endif %}
{%- endif %}
{%- for message in messages %}
{%- if (message.role == "user") or (message.role == "system" and not loop.first) or (message.role == "assistant" and not message.tool_calls) %}
{{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
{%- elif message.role == "assistant" %}
{{- '<|im_start|>' + message.role }}
{%- if message.content %}
{{- '\n' + message.content }}
{%- endif %}
{%- for tool_call in message.tool_calls %}
{%- if tool_call.function is defined %}
{%- set tool_call = tool_call.function %}
{%- endif %}
{{- '\n<tool_call>\n{"name": "' }}
{{- tool_call.name }}
{{- '", "arguments": ' }}
{{- tool_call.arguments | tojson }}
{{- '}\n</tool_call>' }}
{%- endfor %}
{{- '<|im_end|>\n' }}
{%- elif message.role == "tool" %}
{%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
{{- '<|im_start|>user' }}
{%- endif %}
{{- '\n<tool_response>\n' }}
{{- message.content }}
{{- '\n</tool_response>' }}
{%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
{{- '<|im_end|>\n' }}
{%- endif %}
{%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
{{- '<|im_start|>assistant\n' }}
{%- endif %}
"""
```

In [ ]:
def apply_chat_template(
    example: dict[str, list[dict[str, str]]],
    tokenizer: PreTrainedTokenizerBase,
    tools=None,
):
    """
    Apply the tokenizer's chat template to a conversational or completion example.

    Supported keys: "messages", "prompt", "completion", "label".
    Returns a dict with "text" (for messages), "prompt", "completion", and/or "label".
    """
    output = {}

    # Full conversation -> single formatted string
    if "messages" in example:
        output["text"] = tokenizer.apply_chat_template(
            example["messages"], tools=tools, tokenize=False
        )

    # Prompt with generation prompt or continuation
    if "prompt" in example:
        last_role = example["prompt"][-1]["role"]
        add_generation_prompt = last_role == "user"
        continue_final_message = last_role == "assistant"

        if last_role not in ("user", "assistant"):
            raise ValueError(f"Invalid role in the last message: {last_role}")

        prompt = tokenizer.apply_chat_template(
            example["prompt"],
            tools=tools,
            continue_final_message=continue_final_message,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
        output["prompt"] = prompt

    # Completion: extract by removing the prompt prefix
    if "completion" in example:
        prompt_completion = tokenizer.apply_chat_template(
            example["prompt"] + example["completion"], tools=tools, tokenize=False
        )
        output["completion"] = prompt_completion[len(output["prompt"]):]

    if "label" in example:
        output["label"] = example["label"]

    return output


def is_conversational(example: dict) -> bool:
    """Check if an example uses the conversational format (list of role/content dicts)."""
    for key in ("prompt", "chosen", "rejected", "completion", "messages"):
        if key in example:
            val = example[key]
            if isinstance(val, list) and val and isinstance(val[0], dict):
                if "role" in val[0] and "content" in val[0]:
                    return True
    return False

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

dataset = load_dataset("HuggingFaceH4/ultrachat_200k")
dataset = dataset.remove_columns("prompt")
del dataset["train_gen"]
del dataset["test_gen"]

# Small subset for demonstration
sample_size = int(len(dataset["train_sft"]) * 0.001)
dataset["train_sft"] = dataset["train_sft"].select(range(sample_size))
dataset["test_sft"] = dataset["test_sft"].select(range(sample_size))

dataset = dataset.map(
    lambda x: apply_chat_template(x, tokenizer), batched=True
)

print(dataset["train_sft"]["messages"][100])
print("\n")
print(dataset["train_sft"]["text"][100][:1000])

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[{'content': 'Write a descriptive paragraph in a sensory-rich style that captures the unique taste, texture, and aroma of your favorite type of ice cream. Include vivid descriptions that convey the specific flavors, colors, and ingredients of your chosen ice cream flavor. Use sensory language to help your readers imagine how your ice cream feels in the mouth, smells on the tongue, and looks in the bowl. Consider adding personal anecdotes or memories associated with your favorite flavor to create a more engaging reading experience. Make sure to use descriptive adjectives, strong verbs, and figurative language to make your paragraph more captivating and evocative.', 'role': 'user'}, {'content': "As I scoop a spoonful of my all-time favorite ice cream, the rich aroma of fresh coffee beans and creamy chocolate wafts up from the bowl. The chocolate ice cream is as dark as the night sky, sprinkled with tiny flecks of crunchy toffee that adds a perfect texture to each bite. As I take a small 

## 2.2 Pre-Processing

### The Problem: Padding Wastes GPU Memory

In a batch, all sequences must be the **same length**. Without packing, short sequences get padded to match the longest one:
```
Sequence A: [tok, tok, tok]              → 3 tokens
Sequence B: [tok, tok, tok, tok, tok]    → 5 tokens
Sequence C: [tok, tok]                   → 2 tokens

After padding (to length 5):
A: [tok, tok, tok, PAD, PAD]    → 40% wasted
B: [tok, tok, tok, tok, tok]    → 0% wasted
C: [tok, tok, PAD, PAD, PAD]    → 60% wasted
```

> On average, **30-50% of your GPU memory is wasted on PAD tokens** that contribute nothing to training.

---

### The Solution: Pack Sequences Together

Instead of padding, **concatenate sequences into fixed-length chunks**:
```
Before packing (3 sequences, variable length):
A: [1, 2, 3]
B: [4, 5, 6, 7]
C: [8]

After packing (seq_length=5):
Chunk 1: [1, 2, 3, 4, 5]    ← A + start of B
Chunk 2: [6, 7, 8]          ← rest of B + C

No padding needed! Every token is a real token.
```

---

### 🤔 "But wait — doesn't this confuse the model?"

This is the most common concern: if you glue Sequence A and Sequence B together, won't the model think they're one continuous sentence?

**For causal language models (GPT, LLaMA, etc.), this actually doesn't matter much.** Here's why:

> **Causal LM only looks LEFT.** Each token can only attend to tokens **before** it, never after. So when the model processes the first token of Sequence B, it can technically see the end of Sequence A — but this is essentially just noise, similar to how the model sees random context during pretraining.
```
Packed: [A1, A2, A3, B1, B2, B3, B4]
                      ↑
                      B1 can see A1, A2, A3 (irrelevant context)
                      But B2 can see B1 ✓
                      And B3 can see B1, B2 ✓
                      The useful signal (within-sequence) dominates
```

**Why this works in practice:**

1. **Pretraining already handles this.** During pretraining, documents are concatenated together in exactly the same way. The model has already learned to deal with boundaries between unrelated text.

2. **Loss masking helps.** If you're using completion-only loss masking, you're only computing loss on response tokens anyway. The boundary between two packed sequences typically falls between one example's `### End` and the next example's `### Instruction:` — the model isn't penalized for predicting across that boundary.

3. **The benefit outweighs the noise.** In practice, packing lets you train on **2-3x more tokens per batch** with the same GPU memory. The tiny amount of cross-sequence attention noise is negligible compared to this efficiency gain.

---

### ⚠️ When packing can be a problem

For **bidirectional models** (BERT, encoder-decoder) or when using **attention masking between sequences**, packing requires extra care — you'd need a **block-diagonal attention mask** to prevent sequences from attending to each other. This is more complex to implement.
```
Causal LM (GPT/LLaMA):   packing works out of the box ✅
Bidirectional (BERT):     needs special attention mask ⚠️
Encoder-Decoder (T5):     needs special attention mask ⚠️
```

---

> 💡 **Summary:** Packing saves massive amounts of GPU memory by eliminating padding. It works because causal LMs already handle arbitrary context boundaries during pretraining — a few irrelevant tokens from a neighboring sequence is just noise the model has learned to ignore.


In [ ]:
def formatting_prompts_func(example):
    """Format instruction/output pairs into a single string for language modeling."""
    output_texts = []
    for i in range(len(example["instruction"])):
        text = f"### Question: {example['instruction'][i]}\n ### Answer: {example['output'][i]}"
        output_texts.append(text)
    return output_texts


def tokenize(example, dataset_text_field="text"):
    """Tokenize the text field of each example.
    # Input: {"text": str} -> Output: {"input_ids": (seq_len,), "attention_mask": (seq_len,)}
    """
    return tokenizer(example[dataset_text_field])


def pack_examples(examples: dict[str, list[list]], seq_length: int) -> dict[str, list[list]]:
    """
    Data packing: concatenate all sequences then re-chunk into fixed-length blocks.
    This avoids wasting memory on padding short sequences.

    Example:
        input_ids: [[1,2,3], [4,5,6,7], [8]] with seq_length=5
        -> [[1,2,3,4,5], [6,7,8]]

    # Input: variable-length lists -> Output: lists of length seq_length (last may be shorter)
    """
    # Flatten all sequences into one long list
    examples = {k: sum(v, []) for k, v in examples.items()}
    # Re-chunk into blocks of seq_length
    examples = {
        k: [v[i : i + seq_length] for i in range(0, len(v), seq_length)]
        for k, v in examples.items()
    }
    return examples


def truncate(example, max_seq_length):
    """Truncate input_ids and attention_mask to max_seq_length.
    # Input: (seq_len,) -> Output: (min(seq_len, max_seq_length),)
    """
    return {key: example[key][:max_seq_length] for key in ["input_ids", "attention_mask"]}

In [ ]:
# --- Example: tokenize, pack, and truncate ---
print(f"Is Conversational: {is_conversational(dataset['train_sft'][100])}")
dataset = dataset.map(tokenize)
print(dataset["train_sft"]["input_ids"][0][:10])

print("Sample Size Before Packing:", len(dataset["train_sft"]))
dataset = dataset.select_columns(["input_ids", "attention_mask"])
dataset = dataset.map(pack_examples, batched=True, fn_kwargs={"seq_length": 2048})
print("Sample Size After Packing:", len(dataset))

dataset = dataset.map(truncate, fn_kwargs={"max_seq_length": 2048})
print("Final Sequence Length:", len(dataset["train_sft"]["input_ids"][0]))

Is Conversational: True
[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645]
Sample Size Before  207
Sample Size After Packing 2
Final Size:  2048


## 2.3 DataCollator

### 🔍 `DataCollatorForLanguageModeling` vs `DataCollatorForCompletionOnlyLM`

---

Both collators pad sequences and create labels for training. The **key difference** is **which tokens the model learns from**.

---

#### 1️⃣ `DataCollatorForLanguageModeling` — Learn from EVERYTHING

The model is trained to predict **every token** in the sequence. Labels are simply a copy of `input_ids`, with only padding tokens masked to `-100`.
```
Input:  "### Question: What is 2+2? ### Answer: 4."

input_ids: [Question, What, is, 2+2, ?, Answer, 4,  ., PAD, PAD]
labels:    [Question, What, is, 2+2, ?, Answer, 4,  ., -100, -100]
            ^^^^^^^^^^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^     ^^^^^^^^^^
            loss on question ✓        loss on answer ✓   padding ignored

Loss computed on: ALL real tokens
```

> The model learns to predict the question AND the answer. This is standard **causal language modeling** — the same objective used in pretraining.

---

#### 2️⃣ `DataCollatorForCompletionOnlyLM` — Learn ONLY from the response

Inherits from `DataCollatorForLanguageModeling` but adds an **extra step**: after creating labels, it finds the instruction/response boundaries and **masks all instruction tokens to `-100`**.
```
Input:  "### Question: What is 2+2? ### Answer: 4."

input_ids: [Question, What, is, 2+2, ?, Answer, 4,  ., PAD, PAD]
labels:    [-100,    -100,-100,-100,-100, -100,  4,  ., -100, -100]
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^  ^^^^^  ^^^^^^  ^^^^^^^^^^
            question MASKED (ignored)     answer MASKED  padding ignored
                                          template       

Loss computed on: ONLY "4."
```

> The model still **sees** the question (in `input_ids`), but is only **trained to predict** the answer (in `labels`). This is the core idea behind instruction tuning — teach the model **what to answer**, not how to repeat the question.

---

#### 🔄 How the masking works — Two Modes

`DataCollatorForCompletionOnlyLM` supports two modes depending on what templates you provide:

### Mode A: Only `response_template` provided

Finds the response boundary and masks **everything before it**.
```python
collator = DataCollatorForCompletionOnlyLM(
    response_template="### Answer:",
    tokenizer=tokenizer,
)
```
```
"##### Question: What is 2+2? ### Answer: 4."

Step 1: Find "### Answer:" in the token sequence
Step 2: Mask everything up to and including "### Answer:"

labels: [-100, -100, -100, -100, -100, -100, 4, .]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
         everything before response = masked
```

> ⚠️ This mode finds the **last** occurrence of the response template. Works for **single-turn** conversations.

---

##### Mode B: Both `instruction_template` and `response_template` provided

Finds **all** instruction and response boundaries. Masks every instruction segment, keeps every response segment. This supports **multi-turn** conversations.
```python
collator = DataCollatorForCompletionOnlyLM(
    response_template="### Answer:",
    instruction_template="### Question:",
    tokenizer=tokenizer,
)
```
```
"##### Question: What is 2+2? ### Answer: 4. ### Question: And 3+3? ### Answer: 6."

Step 1: Find ALL "### Question:" positions  → [0, 10]
Step 2: Find ALL "### Answer:" positions    → [6, 16]
Step 3: Mask instruction segments, keep response segments

labels: [-100, -100, -100, -100, -100, -100, 4, ., -100, -100, -100, -100, -100, -100, -100, -100, 6, .]
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
         Turn 1 question (masked)                  Turn 2 question (masked)
                                              ^^^^                                                ^^^^
                                              Turn 1 answer (KEPT)                                Turn 2 answer (KEPT)
```

> ✅ This mode handles **multiple turns** — each instruction segment is masked, each response segment is kept.

---

#### 📊 Summary

| | `DataCollatorForLanguageModeling` | `DataCollatorForCompletionOnlyLM` |
|--|-----------------------------------|-----------------------------------|
| **Loss on prompt/question** | ✅ Yes | ❌ No (masked to `-100`) |
| **Loss on response/answer** | ✅ Yes | ✅ Yes |
| **Loss on padding** | ❌ No | ❌ No |
| **Multi-turn support** | N/A | ✅ Yes (Mode B) |
| **Use case** | Pretraining, general LM | Instruction tuning, SFT |
| **What the model learns** | Predict everything | Predict only the response |

In [ ]:
class DataCollatorForLanguageModeling:
    """
    Standard causal LM collator.
    - Pads sequences to equal length
    - Labels = input_ids (model learns to predict EVERY token)
    - Padding positions masked to -100 (ignored by loss)

    Input:  list of dicts [{"input_ids": [...], "attention_mask": [...]}, ...]
    Output: {"input_ids": (batch, seq_len),
             "attention_mask": (batch, seq_len),
             "labels": (batch, seq_len)}
    """

    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, examples):
        input_ids = [e["input_ids"] for e in examples]
        attention_mask = [e["attention_mask"] for e in examples]

        # --- Step 1: Pad to max length in batch ---
        max_len = max(len(seq) for seq in input_ids)

        # Round up to multiple of 8 for GPU efficiency
        if self.pad_to_multiple_of:
            remainder = max_len % self.pad_to_multiple_of
            if remainder != 0:
                max_len += self.pad_to_multiple_of - remainder

        # Pad each sequence: real tokens on left, padding on right
        input_ids = [seq + [self.tokenizer.pad_token_id] * (max_len - len(seq)) for seq in input_ids]
        attention_mask = [seq + [0] * (max_len - len(seq)) for seq in attention_mask]

        # --- Step 2: Create labels ---
        # Labels = copy of input_ids, with padding masked to -100
        # Example:
        #   input_ids: [Hello, world, !, PAD, PAD]
        #   labels:    [Hello, world, !, -100, -100]
        labels = []
        for seq in input_ids:
            label = [tok if tok != self.tokenizer.pad_token_id else -100 for tok in seq]
            labels.append(label)

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels),
        }


class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    """
    Completion-only collator — inherits from DataCollatorForLanguageModeling.
    Extra step: masks instruction/prompt tokens so the model ONLY learns
    to predict the response/answer tokens.

    Two modes:
      Mode A (response_template only):  masks everything before the response
      Mode B (both templates):          handles multi-turn conversations

    Input:  same as parent
    Output: same as parent, but labels has prompt positions set to -100
    """

    def __init__(self, tokenizer, response_template, instruction_template=None, pad_to_multiple_of=8):
        super().__init__(tokenizer, pad_to_multiple_of)

        self.response_template = response_template
        self.instruction_template = instruction_template

        # Pre-encode templates to token IDs for matching
        self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
        self.instruction_token_ids = (
            tokenizer.encode(instruction_template, add_special_tokens=False)
            if instruction_template else None
        )

    def _find_all_occurrences(self, sequence, pattern):
        """Find all starting indices where pattern appears in sequence."""
        positions = []
        for i in range(len(sequence) - len(pattern) + 1):
            if sequence[i : i + len(pattern)] == pattern:
                positions.append(i)
        return positions

    def __call__(self, examples):
        # --- Step 1: Use parent to pad + create base labels ---
        batch = super().__call__(examples)
        labels = batch["labels"].clone()

        for i in range(len(examples)):
            seq = batch["input_ids"][i].tolist()

            if self.instruction_template is None:
                # ── Mode A: only response_template ──
                # Find LAST occurrence of response template, mask everything before it
                #
                # "### Question: What is 2+2? ### Answer: 4."
                #  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                #  mask all of this (including "### Answer:") → -100
                #                                              ^^^^
                #                                              keep only this

                positions = self._find_all_occurrences(seq, self.response_token_ids)

                if not positions:
                    labels[i, :] = -100  # no response found, mask everything
                else:
                    last_pos = positions[-1]
                    end = last_pos + len(self.response_token_ids)
                    labels[i, :end] = -100

            else:
                # ── Mode B: both instruction + response templates ──
                # For multi-turn: mask ALL instruction segments, keep ALL response segments
                #
                # "### Q: What is 2+2? ### A: 4. ### Q: And 3+3? ### A: 6."
                #  ^^^^^^^^^^^^^^^^^^^^^^^^       ^^^^^^^^^^^^^^^^^^^^^^^
                #  Turn 1 instruction → -100      Turn 2 instruction → -100
                #                        ^^^^                            ^^^^
                #                        keep                            keep

                human_positions = self._find_all_occurrences(seq, self.instruction_token_ids)
                response_positions = self._find_all_occurrences(seq, self.response_token_ids)

                if not human_positions or not response_positions:
                    labels[i, :] = -100
                    continue

                # Get the end position of each response template (where actual answer starts)
                response_ends = [pos + len(self.response_token_ids) for pos in response_positions]

                # Mask from start to first response end
                labels[i, : response_ends[0]] = -100

                # Mask each subsequent instruction → response segment
                for h_start, r_end in zip(human_positions[1:], response_ends[1:]):
                    labels[i, h_start:r_end] = -100

                # Mask trailing instruction with no response
                if len(human_positions) > len(response_positions):
                    labels[i, human_positions[-1] :] = -100

        batch["labels"] = labels
        return batch

In [ ]:
# --- Example: basic language modeling collator ---
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
batch = data_collator(torch.Tensor(dataset["train_sft"]["input_ids"][:8]))
print(batch.keys())

dict_keys(['input_ids', 'labels'])


/tmp/ipython-input-1886600918.py:43: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  examples = [torch.tensor(e) for e in examples]


In [ ]:
# --- Example: completion-only collator (loss only on assistant response) ---
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

response_template = " ### Answer:"
instruction_template = "### Question:"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    instruction_template=instruction_template,
    tokenizer=tokenizer,
    mlm=False,
)

text = "### Question: What is 2+2? ### Answer: 2+2 equals 4."
tokens = tokenizer(text, return_tensors="pt")
print(f"Token IDs: {tokens['input_ids'][0].tolist()}")
batch = collator([{"input_ids": tokens["input_ids"][0].tolist()}])

print(f"Labels: {batch['labels'][0].tolist()}")

# Show which tokens contribute to the loss (not masked to -100)
loss_tokens = []
for i, label in enumerate(batch["labels"][0]):
    if label != -100:
        token = tokenizer.decode([batch["input_ids"][0][i]])
        loss_tokens.append((i, token, label.item()))
print(f"Tokens that contribute to loss: {loss_tokens}")

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Token IDs: [14374, 15846, 25, 3555, 374, 220, 17, 10, 17, 30, 16600, 21806, 25, 220, 17, 10, 17, 16819, 220, 19, 13]
Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 220, 17, 10, 17, 16819, 220, 19, 13]
Tokens that contribute to loss: [(13, ' ', 220), (14, '2', 17), (15, '+', 10), (16, '2', 17), (17, ' equals', 16819), (18, ' ', 220), (19, '4', 19), (20, '.', 13)]


## 2.4 Trainer

In [ ]:
class SimpleSFTTrainer:

    def __init__(
        self,
        model,
        tokenizer,
        train_dataset,
        eval_dataset=None,
        batch_size=4,
        learning_rate=5e-5,
        num_epochs=1,
        completion_only_loss=True,
        response_template=" ### Answer:",
        instruction_template="### Question:",
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.num_epochs = num_epochs

        # --- Optimizer ---
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

        # --- Pick the right collator ---
        # completion_only_loss=True  → only train on response tokens
        # completion_only_loss=False → train on all tokens (standard LM)
        if completion_only_loss:
            collator = DataCollatorForCompletionOnlyLM(
                tokenizer=tokenizer,
                response_template=response_template,
                instruction_template=instruction_template,
            )
        else:
            collator = DataCollatorForLanguageModeling(tokenizer=tokenizer)

        # --- DataLoaders ---
        self.train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator
        )
        self.eval_loader = (
            DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)
            if eval_dataset else None
        )

    def train(self):
        """
        The core training loop.

        Each batch goes through:
          input_ids (batch, seq_len)
            → embedding layer     → (batch, seq_len, embed_dim)
            → transformer layers  → (batch, seq_len, model_dim)
            → lm_head             → (batch, seq_len, vocab_size)
            → cross_entropy loss  → scalar
        """
        self.model.train()

        for epoch in range(self.num_epochs):
            total_loss = 0
            num_batches = 0

            for batch_idx, batch in enumerate(tqdm(self.train_dataloader, desc=f"Epoch {epoch+1}")):

                # 1) Move batch to GPU
                batch = {k: v.to(self.model.device) for k, v in batch.items()}

                # 2) Forward pass — model computes logits + loss internally
                #    The model handles label shifting: logits[:-1] vs labels[1:]
                outputs = self.model(**batch)
                loss = outputs.loss

                # 3) Backward pass — compute gradients
                loss.backward()

                # 4) Update weights
                self.optimizer.step()
                self.optimizer.zero_grad()

                total_loss += loss.item()
                num_batches += 1

                if batch_idx % 10 == 0:
                    print(f"  Batch {batch_idx}, Loss: {loss.item():.4f}")

            avg_loss = total_loss / num_batches
            print(f"Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}")

    def evaluate(self):
        """Run evaluation — same forward pass but no gradient computation."""
        if not self.eval_loader:
            print("No evaluation dataset provided.")
            return None

        self.model.eval()
        total_loss = 0
        num_batches = 0

        with torch.no_grad():  # disable gradients for speed + memory
            for batch in tqdm(self.eval_loader, desc="Evaluating"):
                batch = {k: v.to(self.model.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                total_loss += outputs.loss.item()
                num_batches += 1

        avg_loss = total_loss / num_batches
        print(f"Eval Loss: {avg_loss:.4f}")
        return avg_loss

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype=torch.bfloat16
)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_e

In [ ]:
test_data = [
    {"text": "### Question: What is 2+2? ### Answer: 2+2 equals 4."},
    {"text": "### Question: What is the capital of France? ### Answer: The capital of France is Paris."},
    {"text": "### Question: What is photosynthesis? ### Answer: Photosynthesis is the process by which plants convert sunlight into energy."},
]

test_dataset = Dataset.from_list(test_data)
test_dataset = test_dataset.map(tokenize, batched=True)

# Verify the completion-only collator masks instruction tokens
collator = DataCollatorForCompletionOnlyLM(
    response_template=" ### Answer:",
    instruction_template="### Question:",
    tokenizer=tokenizer,
    mlm=False,
)

batch = collator([test_dataset["input_ids"][0], test_dataset["input_ids"][1]])
print(f"Batch keys: {batch.keys()}")
# Input: 2 variable-length sequences -> Output: (2, max_seq_len)
print(f"Input shape: {batch['input_ids'].shape}")
print(f"Labels shape: {batch['labels'].shape}")
print(f"Masked tokens (-100): {(batch['labels'] == -100).sum().item()}")
print(f"Loss tokens: {(batch['labels'] != -100).sum().item()}")

Column(['### Question: What is 2+2? ### Answer: 2+2 equals 4.', '### Question: What is the capital of France? ### Answer: The capital of France is Paris.', '### Question: What is photosynthesis? ### Answer: Photosynthesis is the process by which plants convert sunlight into energy.'])

In [ ]:
trainer = SimpleSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=test_dataset,
    max_seq_length=64,
    batch_size=1,
    learning_rate=1e-4,
    num_epochs=1,
    completion_only_loss=True,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
prepared_dataset = trainer.prepare_dataset(test_dataset, is_conversational_data=False)
# trainer.train()  # uncomment to run training

Model parameters: 494,032,768
Preparing dataset with 3 examples...
Tokenizing dataset...


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Packing examples...


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating sequences...


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset preparation complete. Final size: 2


# 🔄 Label Right-Shifting — Handled by the Model, Not the Collator

---

A common question: **"Do I need to shift my labels so that position `i` predicts position `i+1`?"**

**No.** The data collator outputs `labels = input_ids` (same length, same positions). The **model itself** handles the right-shift internally during loss computation.

---

### Where it happens

This code lives inside the model's `forward()` method (e.g., `transformers/models/qwen2/modeling_qwen2.py`). Every HuggingFace causal LM model has this same pattern:
```python
if labels is not None:
    # Shift so that token at position i predicts token at position i+1
    shift_logits = logits[..., :-1, :].contiguous()   # drop last
    shift_labels = labels[..., 1:].contiguous()         # drop first

    # Flatten for cross-entropy: (batch × seq_len) predictions vs targets
    loss_fct = CrossEntropyLoss()
    shift_logits = shift_logits.view(-1, self.config.vocab_size)
    shift_labels = shift_labels.view(-1)

    loss = loss_fct(shift_logits, shift_labels)
```

---

### Step-by-step example
```
What the collator passes in (NO shifting):
  input_ids: [The, cat, sat, on, mat]
  labels:    [The, cat, sat, on, mat]

What the model computes:
  logits:    [pred_cat, pred_sat, pred_on, pred_mat, pred_EOS]

After shifting inside the model:
  shift_logits: [pred_cat, pred_sat, pred_on, pred_mat]    # logits[:-1]
  shift_labels: [cat,      sat,      on,      mat]          # labels[1:]
                  ↑          ↑         ↑         ↑
                  position 0  position 1 position 2 position 3
                  predicts    predicts   predicts   predicts
                  next token  next token next token next token

Loss = CrossEntropy(shift_logits, shift_labels)
```

---

### What about `-100` masked tokens?

The shift preserves the `-100` masks from the collator. `CrossEntropyLoss` has `ignore_index=-100` **by default**, so masked positions contribute zero to the loss — no extra configuration needed.
```
With completion-only masking:
  labels:        [-100, -100, -100, -100, sat, on, mat]
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^
                  prompt (masked by collator)

  After shift:
  shift_labels:  [-100, -100, -100, sat, on, mat]
                  ^^^^^^^^^^^^^^^^^^^^^
                  still masked → ignored by CrossEntropyLoss
                                       ^^^^^^^^^^^^^
                                       loss computed only here ✓
```

---

> 💡 **Key takeaway:** Just pass `labels = input_ids` with prompt/padding positions set to `-100`. The model handles the alignment internally. You never need to do `labels = input_ids[1:]` yourself.